# Agentic RAG Explained
## Query rewriting, routing, and self-checking with LangGraph

A beginner-friendly walkthrough. You will build one small prototype from start to finish, then compare it with ordinary, one-shot RAG on the same data.

**What you will learn**

1. What agentic RAG is, in plain English, before any code
2. How to build a **naive RAG** baseline with citations
3. How **query rewriting** fixes messy wording, like "pro plan" or "file cap"
4. How **routing** picks the right document store
5. How **multi-step retrieval** and **self-checking** catch thin evidence
6. How to generate **grounded answers** that cite their sources
7. How to **evaluate** both systems with real measured numbers
8. When an agent is **overkill**

You only need a free Colab runtime, an Anthropic API key in Colab Secrets, and this notebook.

The code lives in this repo: `https://github.com/davolu/agentic-rag-explained`


## 1. Setup

We install a few libraries, load the API key safely (we never print it), and set the model with a one-line swap.


In [ ]:
# On Colab, install dependencies first. On a local machine, use requirements.txt.
import sys, subprocess
pkgs = ["anthropic", "langgraph", "sentence-transformers", "numpy"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
print("Libraries ready.")


In [ ]:
import os, sys, subprocess
from pathlib import Path

# Find the repo root (works in Colab after a clone, and in this project folder).
ROOT = Path.cwd()
if not (ROOT / "src" / "agentic_rag_demo.py").exists():
    repo = Path("agentic-rag-explained")
    if not repo.exists():
        subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/davolu/agentic-rag-explained.git"])
    ROOT = repo.resolve()
    os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

from agentic_rag_demo import load_key, get_client, PROVIDER, MODEL
ok = load_key()
print("API key found:", ok)
print("Provider:", PROVIDER)
print("Model:", MODEL)
print("The key itself is never printed.")
print("To swap the Claude model, change MODEL in src/agentic_rag_demo.py.")
print("To swap provider, change PROVIDER to 'openai' and set OPENAI_API_KEY.")
if not ok:
    raise SystemExit("Add ANTHROPIC_API_KEY to Colab Secrets (key icon) and enable notebook access.")


## 2. An analogy, before any retrieval code

**Naive RAG is a clerk at a front desk.** You ask a question, and they photocopy the three pages whose wording is closest to yours. That is often enough when the answer sits in one paragraph, and you used the official names.

**Agentic RAG is a researcher with a checklist.** First they rephrase your question clearly. Then they pick the right drawer — product, billing, or policy. Then they read. Then they check: did these pages actually answer you? If not, they go back and look again.

We will build both, on the same tiny help-desk dataset, and then keep score with real numbers.


## 3. The demo world

**Alderpeak** is a fictional company. Its help desk product is called **Ticketly**. There are 12 short documents, split into three stores: **product**, **billing**, and **policy**. A few traps are built in on purpose: people say "pro plan" when the file says **Growth**, they say "file cap" when the file says **attachment storage**, and **Flare** is both a free live feed and a paid add-on called **Flare Reports**.


In [ ]:
from agentic_rag_demo import load_docs

docs = load_docs()
print(f"{len(docs)} documents, in three stores\n")
for store in ("product", "billing", "policy"):
    print(f"=== {store.upper()} ===")
    for d in docs:
        if d["store"] != store:
            continue
        preview = d["text"].replace("\n", " ")
        if len(preview) > 100:
            preview = preview[:100] + "..."
        print(f"{d['id']:6}  {d['title']}")
        print(f"        {preview}\n")


In [ ]:
print("Wording traps you will see:")
print("  'pro plan' or 'mid tier'  vs  the official name, Growth")
print("  'file cap'                vs  the attachment storage pool")
print("  Flare, the live inbox feed (included)  vs  Flare Reports ($29 / month)")
print("  A service credit needs two facts: the outage duration, and the policy")


## 4. A naive RAG baseline

Ordinary RAG does one thing: embed the question, fetch the nearest chunks, and ask the model to answer only from those chunks. We will try a simple question, then two harder ones.


In [ ]:
from agentic_rag_demo import chunk_docs, embed_texts, vector_search, format_context, naive_ask, answer_from_context

client = get_client()
chunks = chunk_docs(docs)
matrix = embed_texts([c["text"] for c in chunks])
print(f"{len(chunks)} chunks embedded, shape {tuple(matrix.shape)}")


In [ ]:
def ask_naive(question, k=3):
    hits = vector_search(question, chunks, matrix, k=k)
    print("Retrieved:")
    for h in hits:
        print(f"  {h['doc_id']}  store={h['store']:8}  score={h['score']:.3f}  {h['title']}")
    result = answer_from_context(client, question, format_context(hits))
    print("\nAnswer:\n" + result["answer"])
    print(f"\nCitations: {result['citations']}   latency: {result['latency_s']} s")
    return hits, result

simple_q = "How do I turn on SSO in Ticketly?"
simple_hits, simple_naive = ask_naive(simple_q)


In [ ]:
rewrite_q = "Why is my team hitting the file cap on the pro plan?"
rewrite_hits, rewrite_naive = ask_naive(rewrite_q)


In [ ]:
hop_q = "What credit can a Growth customer claim for last Monday's API outage?"
hop_hits, hop_naive = ask_naive(hop_q)


A simple how-to question works. The other two are harder: one uses unofficial wording, and one needs facts from two documents. Grounded naive RAG should say it does not know, rather than guess.


## 5. The agent graph

The agent is a small LangGraph state machine. Each box is a step: rewrite the question, pick a store, retrieve, check the evidence, and only then write the answer. If the self-check fails, the graph loops back and looks again, up to two extra tries.


In [ ]:
from agentic_rag_demo import build_agent, graph_mermaid

app = build_agent(client, chunks, matrix, k=3)
print("Graph nodes: rewrite → route → retrieve → grade → generate")
print("If the self-check fails, grade loops back to rewrite.")
print()
print(graph_mermaid(app))


## 6. Query rewriting and routing

First the agent rewrites the messy question in official language. Then it picks one or more stores: product, billing, or policy.


In [ ]:
from agentic_rag_demo import rewrite_query, route_query

print("Original:", rewrite_q)
rw = rewrite_query(client, rewrite_q)
print("Rewritten:", rw["rewritten"])
rt = route_query(client, rw["rewritten"])
print("Routes:", ", ".join(rt["stores"]))


## 7. Multi-step retrieval and self-checking

Now we run the full agent on the multi-hop question. Watch the steps. If the first batch of pages does not include the outage duration, the self-check should send the agent back for another look.


In [ ]:
from agentic_rag_demo import run_agent

hop_agent = run_agent(app, hop_q)
print("Steps:")
for i, step in enumerate(hop_agent["steps"], start=1):
    print(f"  {i}. {step}")
print(f"\nTotal latency: {hop_agent['total_latency_s']} s")


## 8. Grounded answers

The model may only use what we retrieved, and it must cite the source documents. Let's compare both systems on that same outage question.


In [ ]:
print("=== Naive RAG ===")
print(hop_naive["answer"])
print(f"Citations: {hop_naive['citations']}   latency: {hop_naive['latency_s']} s")
print()
print("=== Agentic RAG ===")
print(hop_agent["answer"])
print(f"Citations: {hop_agent['citations']}   latency: {hop_agent['total_latency_s']} s")


## 9. Evaluation against the naive baseline

We score both systems on the same eight questions. A deterministic checker looks for required phrases. Latency is measured, not estimated. The numbers below are from the recorded run in `run/results/`.


In [ ]:
import json
from agentic_rag_demo import load_eval_questions, score_answer, naive_ask

summary_path = ROOT / "run" / "results" / "summary.json"
rows_path = ROOT / "run" / "results" / "rows.json"

if summary_path.exists() and rows_path.exists():
    summary = json.loads(summary_path.read_text())
    eval_rows = json.loads(rows_path.read_text())
    print("Using the measured run in run/results/.")
else:
    questions = load_eval_questions()
    eval_rows = []
    for item in questions:
        q = item["question"]
        naive = naive_ask(client, q, chunks, matrix, k=3)
        agent = run_agent(app, q)
        eval_rows.append({
            "id": item["id"], "kind": item["kind"], "question": q, "gold": item["gold"],
            "naive": score_answer(item, naive), "agentic": score_answer(item, agent),
        })
    def _sum(key):
        xs = [r[key] for r in eval_rows]; n = len(xs)
        return {"n": n, "correct": sum(x["correct"] for x in xs),
                "citation_ok": sum(x["citation_ok"] for x in xs),
                "accuracy": round(sum(x["correct"] for x in xs) / n, 3),
                "citation_accuracy": round(sum(x["citation_ok"] for x in xs) / n, 3),
                "avg_latency_s": round(sum(x["latency_s"] for x in xs) / n, 3)}
    summary = {"naive": _sum("naive"), "agentic": _sum("agentic")}

print(f"{'ID':4} {'Kind':10} {'Naive':8} {'Agentic':8} {'N lat':7} {'A lat':7}")
print("-" * 54)
for r in eval_rows:
    print(f"{r['id']:4} {r['kind']:10} {str(r['naive']['correct']):8} {str(r['agentic']['correct']):8} "
          f"{r['naive']['latency_s']:<7} {r['agentic']['latency_s']:<7}")
print()
n, a = summary["naive"], summary["agentic"]
print(f"Naive RAG    {n['correct']}/{n['n']} correct  ({n['accuracy']:.1%})  citations {n['citation_ok']}/{n['n']}  avg {n['avg_latency_s']} s")
print(f"Agentic RAG  {a['correct']}/{a['n']} correct  ({a['accuracy']:.1%})  citations {a['citation_ok']}/{a['n']}  avg {a['avg_latency_s']} s")


## 10. When the agent is overkill

An agent is not always better. For a question that already sits in one paragraph, such as a data-retention policy, one-shot retrieval is simpler and just as accurate. On this run it was also much faster. Choose the tool that fits the question.


In [ ]:
policy_q = "How long is customer data kept after an account is cancelled?"
print("Question:", policy_q, "\n")
print("=== Naive RAG ===")
_, policy_naive = ask_naive(policy_q)
print("\n=== Agentic RAG ===")
policy_agent = run_agent(app, policy_q)
print(policy_agent["answer"])
print(f"Citations: {policy_agent['citations']}   latency: {policy_agent['total_latency_s']} s")


## Recap

You now have a small prototype:

1. A naive RAG baseline — one search, one answer, with citations
2. Query rewriting — messy wording becomes a clear question
3. Routing — pick the product, billing, or policy store
4. Multi-step retrieval and self-checking — go back if the evidence is thin
5. Grounded answers with citations
6. A fair score against the baseline, using real measured numbers

On this dataset, naive RAG got **6 / 8** questions right (75%, average 1.075 s). Agentic RAG got **8 / 8** (100%, average 7.523 s). The extra wins were a wording trap ("file cap" on the "pro plan") and a multi-hop credit question, where the outage duration and the credit policy lived in different documents. The retention question was a tie: the agent was not needed, and it was slower.

The notebook and data are in the GitHub repo linked in the description.

David Oluyale  |  Agentic AI Hub
